# VisionBridge — Phase 2 development evaluation

This separate notebook runs the **existing evaluator**, not training. It preserves run02 and all checkpoints. The first run is a **20-example debugging pilot**, not an accuracy experiment. The full 1,000-example run is disabled.

**Start:** upload this notebook to Colab, select **Runtime → Change runtime type → T4 GPU → Save**, then run the numbered sections in order using each cell's ▶ button. Do not run your training notebook alongside this notebook.

Configurations: **A** BLIP v2 alone; **B** restricted question-guided OCR decisions at 0.90; **C** the same policy at 0.95. All share the same BLIP predictions and cached evidence. No learned answerability model is included.

The Phase 1 **59.48% / 3,319 examples** is a held-out result. This notebook does not evaluate those examples or use them to choose thresholds.


## 1. Prepare the local code ZIP

On your laptop, open **PowerShell** and paste the commands below. They package the current uncommitted Python code; no checkpoints or results are included. If the ZIP already exists, choose another ZIP filename.

```powershell
Set-Location 'C:\Users\Vedant\mai-project'
$bundle = Join-Path $PWD '.deps\visionbridge-phase2-dev-code.zip'
if (Test-Path -LiteralPath $bundle) { throw 'ZIP exists; choose another filename.' }
Compress-Archive -LiteralPath @('config.py', 'evaluate_vqa.py', 'visionbridge_training.py', 'export_phase2_dev.py', 'evaluate_phase2_dev.py', 'requirements-training.txt', 'requirements-phase2-colab.txt', 'modules', 'utils') -DestinationPath $bundle
```

Upload **only that ZIP** in Section 2. Include both complete `modules` and `utils` folders. Do not upload weights or the entire repository. Cloning GitHub alone will not supply these uncommitted files.


## 2. Upload and verify the Python files

Click ▶, then **Choose Files**, and select the code ZIP from Section 1. Files are extracted into a separate Colab folder. Rerunning this cell accepts an identical upload; different existing files are preserved and cause an error. Restart with a fresh runtime if you need different code.


In [ ]:
from pathlib import Path, PurePosixPath
from google.colab import files
import io, json, os, sys, zipfile, hashlib

CODE = Path('/content/visionbridge_phase2_dev_setup_v2')
uploaded = files.upload()
assert len(uploaded) == 1, 'Select only your code ZIP.'
zip_name, zip_bytes = next(iter(uploaded.items()))
assert zip_name.lower().endswith('.zip'), 'Upload the ZIP prepared in Section 1.'
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
    members = {}
    for info in archive.infolist():
        if info.is_dir():
            continue
        name = info.filename.replace('\\', '/')
        path = PurePosixPath(name)
        assert not path.is_absolute() and '..' not in path.parts and ':' not in name, 'Unsafe ZIP path.'
        if path.suffix not in ('.py', '.txt') or '__pycache__' in path.parts:
            continue
        assert name not in members, 'Duplicate ZIP member.'
        assert info.file_size <= 5_000_000, 'Unexpectedly large source file.'
        members[name] = archive.read(info)
    assert sum(map(len, members.values())) < 50_000_000, 'Code ZIP is unexpectedly large.'
    required = ['config.py', 'evaluate_vqa.py', 'visionbridge_training.py',
                'evaluate_phase2_dev.py', 'export_phase2_dev.py',
                'requirements-training.txt', 'requirements-phase2-colab.txt',
                'modules/__init__.py', 'modules/vqa.py', 'modules/ocr.py',
                'modules/object_detection.py', 'modules/evidence_store.py',
                'modules/evidence_decision.py', 'modules/question_intent.py',
                'modules/qges.py', 'modules/fusion.py', 'modules/reliability.py',
                'utils/__init__.py', 'utils/image_utils.py', 'utils/model_loader.py']
    assert all(name in members for name in required), 'Required code missing: include the complete modules/utils folders.'
    # Check all existing files before creating any new ones.
    for name, data in members.items():
        target = CODE / name
        if target.exists():
            assert target.read_bytes() == data, f'Different existing file: {target}. Use a fresh runtime.'
    CODE.mkdir(parents=True, exist_ok=True)
    for name, data in members.items():
        target = CODE / name
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            with target.open('xb') as stream:
                stream.write(data)

os.chdir(CODE)
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
print('Code ready:', CODE, '| ZIP SHA256:', hashlib.sha256(zip_bytes).hexdigest())


## 3. Install dependencies and verify the T4

Click ▶. This installs only the evaluation dependencies. It preserves the **installed Torch, torchvision, NumPy and Pillow versions** with exact constraints; incompatible requirements stop installation instead of replacing those packages. The Phase 1 training requirements are not included.

This cell is safe to rerun. A loaded-version mismatch is printed for diagnosis, not treated as proof that installation changed a package. A restart is requested only when this installation changes an already imported package, or when the fresh-process import/ABI check passes but stale imports make the current kernel fail.

If a genuine restart is requested, use **Runtime → Restart session**, rerun Section 2 with the **updated ZIP**, then rerun this section. Initial downloads need internet access. Do not use the old ZIP: it contains the old NumPy/Pillow restrictions.


In [ ]:
import subprocess
import importlib.metadata as metadata
import uuid

# This cell protects the current Colab environment, not the local app's pins.
protected_packages = ('torch', 'torchvision', 'numpy', 'Pillow')
protected_before = {name: metadata.version(name) for name in protected_packages}
loaded_roots_before = {name.split('.')[0] for name in sys.modules}
installed_before = {
    dist.metadata['Name'].lower(): dist.version
    for dist in metadata.distributions() if dist.metadata['Name']
}
print('Preserving installed versions:', protected_before)
for module_name, package_name in [('numpy', 'numpy'), ('PIL', 'Pillow')]:
    loaded = sys.modules.get(module_name)
    if loaded is not None:
        print(f'{module_name}: loaded={getattr(loaded, "__version__", "unknown")}, '
              f'installed={protected_before[package_name]} (diagnostic only)')

constraints = Path('/content') / ('phase2-constraints-' + uuid.uuid4().hex + '.txt')
with constraints.open('x', encoding='utf-8') as stream:
    stream.write(''.join(f'{name}=={value}\n' for name, value in protected_before.items()))
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    '--upgrade-strategy', 'only-if-needed',
    '-r', str(CODE / 'requirements-phase2-colab.txt'),
    '-c', str(constraints)], check=True)

protected_after = {name: metadata.version(name) for name in protected_packages}
assert protected_after == protected_before, 'A protected package changed unexpectedly; stop here.'

# A restart is justified when THIS installation changes code already imported.
def changed_loaded_packages(before, after, loaded_roots, package_map):
    return sorted({
        package for root in loaded_roots
        for package in package_map.get(root, [])
        if package.lower() in after
        and before.get(package.lower()) != after[package.lower()]
    })

installed_after = {
    dist.metadata['Name'].lower(): dist.version
    for dist in metadata.distributions() if dist.metadata['Name']
}
restart_packages = changed_loaded_packages(
    installed_before, installed_after, loaded_roots_before, metadata.packages_distributions())
if restart_packages:
    raise RuntimeError(
        'Installation changed already imported packages: ' + ', '.join(restart_packages) +
        '. Restart the session once, then rerun Sections 2 and 3 with the updated ZIP. '
        'An unchanged rerun will not trigger this check.')

# Import/ABI check in a fresh Python process: no model loading or evaluation.
health_check = """
import numpy as np
import torch, torchvision, cv2, easyocr, ultralytics
from PIL import Image
from transformers import BlipProcessor, BlipForQuestionAnswering
assert torch.cuda.is_available(), 'Enable the T4 GPU runtime.'
sample = np.asarray(Image.new('RGB', (2, 2))).copy()
assert tuple(torch.from_numpy(sample).shape) == (2, 2, 3)
assert cv2.cvtColor(sample, cv2.COLOR_RGB2GRAY).shape == (2, 2)
print('Import/ABI check passed; GPU:', torch.cuda.get_device_name(0))
"""
probe = subprocess.run(
    [sys.executable, '-c', health_check], cwd=CODE,
    capture_output=True, text=True)
if probe.returncode:
    print(probe.stdout)
    print(probe.stderr)
    raise RuntimeError(
        'Fresh-process dependency/GPU check failed. This is not an automatic restart request; '
        'inspect the error above before continuing.')

try:
    import numpy as np
    import torch, torchvision, cv2, easyocr, ultralytics
    from PIL import Image
    from transformers import BlipProcessor, BlipForQuestionAnswering
    assert torch.cuda.is_available(), 'Enable the T4 GPU runtime.'
    sample = np.asarray(Image.new('RGB', (2, 2))).copy()
    assert tuple(torch.from_numpy(sample).shape) == (2, 2, 3)
    assert cv2.cvtColor(sample, cv2.COLOR_RGB2GRAY).shape == (2, 2)
except Exception as error:
    raise RuntimeError(
        'Fresh Python passed but the current kernel failed its import/ABI check. '
        'Restart the session once to clear stale imports, then rerun Sections 2 and 3. '
        f'Current-kernel error: {error}') from error

print(probe.stdout.strip())
print('Torch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('NumPy:', np.__version__, '| Pillow:', Image.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('Dependencies ready. Continue to Section 4; no evaluation has run.')


## 4. Mount Drive, check run02 and choose a new pilot folder

Click ▶ and approve your own Google Drive mount. The checkpoint path is already set to your verified run02 export. No weights are uploaded or copied.

Leave `RESUME_PILOT = False` and `PILOT_NAME = ''` for a new run. A unique pilot name is generated and printed. After a disconnect, paste that exact printed folder name into `PILOT_NAME`, set `RESUME_PILOT = True`, and rerun setup sections. Resume requires identical code, packages, images and settings.


In [ ]:
from google.colab import drive
from datetime import datetime, timezone

drive.mount('/content/drive')
RUN02 = Path('/content/drive/MyDrive/VisionBridge/runs/vizwiz-5000-lr1e5-run02')
V2 = RUN02 / 'exports/export-001564-fde95532/blip_vizwiz_v2'
PHASE2 = Path('/content/drive/MyDrive/VisionBridge/phase2-dev')
RESUME_PILOT = False
PILOT_NAME = ''  # Only fill with a previous printed pilot name when resuming.

assert (RUN02 / 'split_manifest.json').is_file(), 'Missing run02 split_manifest.json. Do not recreate the split.'
assert (RUN02 / 'official_normalization.json').is_file(), 'Missing saved official scoring rules.'
from visionbridge_training import verify_checkpoint, sha256, guard_output
CHECKPOINT_HASHES = verify_checkpoint(V2)
for name in ('generation_config.json', 'special_tokens_map.json'):
    if (V2 / name).is_file():
        CHECKPOINT_HASHES[name] = sha256(V2 / name)
MANIFEST_HASH = sha256(RUN02 / 'split_manifest.json')
RULES_HASH = sha256(RUN02 / 'official_normalization.json')
assert PHASE2.resolve() != RUN02.resolve() and RUN02.resolve() not in PHASE2.resolve().parents
for checkpoint in (V2, CODE / 'models/blip_vizwiz'):
    guard_output(checkpoint, PHASE2)
PHASE2.mkdir(parents=True, exist_ok=True)
DEV = PHASE2 / 'run02-dev-only.json'
if not PILOT_NAME:
    assert not RESUME_PILOT, 'Supply the exact previous pilot name to resume.'
    PILOT_NAME = 'pilot-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:8]
assert PILOT_NAME.startswith('pilot-') and Path(PILOT_NAME).name == PILOT_NAME and '\\' not in PILOT_NAME and '/' not in PILOT_NAME
PILOT_DIR = PHASE2 / PILOT_NAME
assert PILOT_DIR.exists() if RESUME_PILOT else not PILOT_DIR.exists(), 'Resume folder missing, or new pilot folder already exists.'
MODEL_CACHE = PHASE2 / 'model-cache'
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
EVIDENCE_CACHE = PHASE2 / 'evidence-cache'
print('Verified checkpoint:', V2)
print('Weights SHA256:', CHECKPOINT_HASHES['model.safetensors'])
print('PILOT_NAME:', PILOT_NAME)
print('Results will be saved to:', PILOT_DIR)


## 5. Export and verify the exact development split

Click ▶. This calls `export_phase2_dev.py`; it does not sample new data. An existing dev-only export is checked against run02 before reuse. References, duplicate frequencies, empty slots and answerability labels are preserved.

The saved Phase 1 JSON contains several splits in one container. Only its `dev` field is selected or inspected. No holdout rows are traversed, printed, exported or scored. This check confirms the dev export matches its source; it deliberately does not open held-out IDs to check overlap.


In [ ]:
from export_phase2_dev import export_dev, validate_dev
from modules.evidence_store import digest

if not DEV.exists():
    export_dev(RUN02, DEV)
payload = json.loads(DEV.read_text(encoding='utf-8'))
dev_rows = validate_dev(payload)
saved_dev = json.loads((RUN02 / 'split_manifest.json').read_text(encoding='utf-8'))['dev']
assert dev_rows == saved_dev, 'Export differs from saved dev rows. Preserve it; investigate before continuing.'
assert digest(saved_dev) == payload['rows_sha256']
assert payload['source_manifest_sha256'] == MANIFEST_HASH
assert payload['official_rules'] == json.loads((RUN02 / 'official_normalization.json').read_text(encoding='utf-8'))
assert len(dev_rows) == 1000
assert not any(key in payload for key in ('train', 'holdout', 'validation'))
DEV_FILE_HASH = sha256(DEV)
DEV_ROWS_HASH = digest(dev_rows)
PILOT_ROWS_HASH = digest(dev_rows[:20])
print('Exact saved dev rows:', len(dev_rows), '| checksum:', DEV_ROWS_HASH)
print('Evaluator input:', DEV)
print('Pilot uses the first 20 dev IDs only. The 3,319-example holdout is not an evaluator input.')


## 6. Reuse images and prepare missing development images safely

Click ▶. If all dev images remain in the original Colab data folder, they are used without copying. Otherwise, available dev images are reused by copying into the **separate Phase 2 data folder**; only missing dev IDs are extracted from the validation ZIP. Existing files are never overwritten.

The validation ZIP also contains held-out images, but only named **dev images** are extracted or opened. No annotation download or train/holdout preparation function is called. Downloading the full image archive may take time and several GB of Drive space. You may add your existing image/ZIP paths below.


In [ ]:
import shutil
from visionbridge_training import download_once, extract_named, DATA_BASE

DATA = PHASE2 / 'data'
DATA.mkdir(parents=True, exist_ok=True)
IMAGE_CANDIDATES = [DATA / 'images', Path('/content/visionbridge_data/images')]
ARCHIVE_CANDIDATES = [DATA / 'val.zip', Path('/content/visionbridge_data/val.zip')]
names = [row['image'] for row in dev_rows]
IMAGES = next((folder for folder in IMAGE_CANDIDATES
               if all((folder / name).is_file() for name in names)), None)
if IMAGES is None:
    IMAGES = DATA / 'images'
    IMAGES.mkdir(parents=True, exist_ok=True)
    for name in names:
        target = IMAGES / name
        source = next((folder / name for folder in IMAGE_CANDIDATES
                       if folder != IMAGES and (folder / name).is_file()), None)
        if source is not None:
            if target.exists():
                assert sha256(target) == sha256(source), f'Existing image differs: {target}. Preserve it and investigate.'
            else:
                with source.open('rb') as src, target.open('xb') as dst:
                    shutil.copyfileobj(src, dst)
    missing = [name for name in names if not (IMAGES / name).is_file()]
    if missing:
        archive = next((path for path in ARCHIVE_CANDIDATES if path.is_file()), None)
        if archive is None:
            archive = download_once(DATA_BASE + 'images/val.zip', DATA / 'val.zip')
        assert zipfile.is_zipfile(archive), 'Existing ZIP is incomplete. Preserve it and choose another Phase 2 data path.'
        extract_named(archive, missing, IMAGES)
assert all((IMAGES / name).is_file() for name in names)
# Verify only the dev images; no held-out image is opened.
for name in names:
    with Image.open(IMAGES / name) as image:
        image.verify()
print('Verified development images:', len(names), '| directory:', IMAGES)


## 7. Run the real 20-example pilot

Click ▶ **only after Sections 1–6 succeed**. This is the only enabled evaluation cell. It calls the existing evaluator with `--mode pilot --limit 20`. A, B and C share one prediction/evidence capture per example. Evidence caches and completed captures are stored on Drive.

First-use model loading or OCR/YOLO downloads can make the pilot slower. If it stops, preserve its folder and use the resume settings in Section 4. Do not change OCR thresholds based on this pilot.


In [ ]:
from evaluate_phase2_dev import parser as evaluation_parser, run as evaluate_dev

assert sha256(DEV) == DEV_FILE_HASH and digest(dev_rows) == DEV_ROWS_HASH
pilot_args = [
    '--dev-manifest', str(DEV), '--images', str(IMAGES), '--checkpoint', str(V2),
    '--output', str(PILOT_DIR), '--evidence-cache', str(EVIDENCE_CACHE),
    '--model-cache', str(MODEL_CACHE), '--yolo-model', str(MODEL_CACHE / 'yolov8n.pt'),
    '--device', 'cuda', '--mode', 'pilot', '--limit', '20']
if RESUME_PILOT:
    pilot_args.append('--resume')
PILOT_RESULTS = evaluate_dev(evaluation_parser().parse_args(pilot_args))
print('Pilot complete. Results:', PILOT_RESULTS)


## 8. Verify the pilot and display its comparison

Click ▶. All three policies are displayed; C is labelled redundant if it matches B. Counts and sample score changes are **debugging diagnostics**, not an accuracy claim. Corrected/worsened means higher/lower official VQA agreement for an individual example; corrected does not necessarily mean fully correct.

The latency column is the evaluator's shared-capture estimate. Cache hits and initial model loading affect it; it is not three independently timed model runs.


In [ ]:
from evaluate_phase2_dev import summarize
from IPython.display import display, HTML
from html import escape

summary = json.loads((PILOT_RESULTS / 'summary.json').read_text(encoding='utf-8'))
records = {name: json.loads((PILOT_RESULTS / f'{name}_records.json').read_text(encoding='utf-8'))
           for name in ('A', 'B', 'C')}
identity = json.loads((PILOT_DIR / 'identity.json').read_text(encoding='utf-8'))
assert identity['mode'] == 'pilot' and identity['rows_sha256'] == PILOT_ROWS_HASH
assert identity['dev_manifest_sha256'] == DEV_FILE_HASH
expected_ids = [row['image'] for row in dev_rows[:20]]
for name, values in records.items():
    assert len(values) == 20 and [r['image'] for r in values] == expected_ids
    assert all(r['question'] == row['question'] and r['answers'] == row['answers']
               and r['answerable'] == row['answerable'] for r, row in zip(values, dev_rows[:20]))
assert all(a['baseline_answer'] == b['baseline_answer'] == c['baseline_answer']
           for a, b, c in zip(records['A'], records['B'], records['C']))
assert sha256(RUN02 / 'split_manifest.json') == MANIFEST_HASH
assert sha256(RUN02 / 'official_normalization.json') == RULES_HASH
assert sha256(DEV) == DEV_FILE_HASH
for filename, expected_hash in CHECKPOINT_HASHES.items():
    assert sha256(V2 / filename) == expected_hash, f'Checkpoint changed: {filename}'

# Standard-library HTML table: no additional pandas dependency.
def show_table(headers, rows):
    def cell(value):
        return escape(f'{value:.3f}' if isinstance(value, float) else str(value))
    head = ''.join('<th style="padding:8px;text-align:left">' + cell(h) + '</th>' for h in headers)
    body = ''.join('<tr>' + ''.join('<td style="padding:8px;vertical-align:top">' + cell(v) + '</td>'
                                   for v in row) + '</tr>' for row in rows)
    display(HTML('<table style="border-collapse:collapse;border:1px solid #ccc"><thead><tr>' +
                 head + '</tr></thead><tbody>' + body + '</tbody></table>'))

comparison_rows = []
for name in ('A', 'B', 'C'):
    stats = summarize(records[name], pilot=True)
    label = name + (' (same answers as B)' if name == 'C' and summary['C_redundant_on_this_run'] else '')
    comparison_rows.append([label, stats['examples'], stats['ocr_overrides'], stats['beneficial_overrides'],
                            stats['harmful_overrides'], stats['neutral_overrides'], stats['unchanged_answers'],
                            stats['average_processing_seconds'], stats['extraction_failures'], stats['partial_extractions']])
show_table(['Policy', 'Examples', 'OCR overrides', 'Corrected*', 'Worsened*', 'Neutral overrides',
            'Unchanged', 'Mean seconds/example', 'Failed extraction', 'Partial extraction'], comparison_rows)
print('Verified: same 20 dev rows, references and BLIP candidates for A/B/C; Phase 1 files unchanged.')
print('Pilot only: no aggregate VQA accuracy is reported. Keep 0.90 provisional.')
print('Results saved:', PILOT_RESULTS)


## 9. Inspect OCR changes and supporting evidence

Click ▶ to view changed answers for B and C, followed by five baseline examples. Reference frequencies remain intact in the saved JSON. Full OCR boxes, stable IDs, selection details and extraction status are in `B_records.json` and `C_records.json`.

Review failed/partial extraction and ambiguous cases before considering the full development run. No improved accuracy is claimed here.


In [ ]:
from collections import Counter

def reference_counts(answers):
    from evaluate_vqa import usable_answers
    return '; '.join(f'{answer} × {count}' for answer, count in Counter(usable_answers(answers)).items())

for name in ('B', 'C'):
    changed = [r for r in records[name] if r['ocr_override']]
    print(f'Policy {name}: {len(changed)} OCR overrides')
    if changed:
        show_table(['Image', 'Question', 'References (counts)', 'BLIP', 'Final', 'Outcome',
                    'Baseline score (0–1)', 'Final score (0–1)', 'Supporting IDs', 'Decision reason', 'Seconds'],
                   [[r['image'], r['question'], reference_counts(r['answers']), r['baseline_answer'], r['final_answer'],
                     'Higher agreement' if r['ocr_corrected_blip'] else ('Lower agreement' if r['ocr_worsened_blip'] else 'Same agreement'),
                     r['baseline_vqa_score'], r['final_vqa_score'], ', '.join(r['supporting_evidence_ids']),
                     r['decision_reason'], r['processing_seconds']] for r in changed])
    else:
        print('No changed OCR answers in this pilot; this does not prove that OCR has no value.')
show_table(['Image', 'Question', 'References (counts)', 'BLIP baseline', 'B final', 'B source', 'B reason'],
           [[r['image'], r['question'], reference_counts(r['answers']), r['baseline_answer'],
             r['final_answer'], r['answer_source'], r['decision_reason']] for r in records['B'][:5]])
print('Saved files:', ', '.join(sorted(p.name for p in PILOT_RESULTS.iterdir())))


## 10. Full 1,000-example evaluation — disabled

**Stop here after the pilot and review its diagnostics.** Leave `RUN_FULL_DEV = False`. Running this cell in its default state does nothing. Only change it after approval for the full development evaluation. The full run uses a separate new folder and the same dev-only export/cache; it does not use the held-out set.

For full-run recovery, use the evaluator's documented `--resume` option with the exact printed full-run folder and unchanged settings. Do not rerun this cell to resume: it intentionally generates a fresh folder.


In [ ]:
RUN_FULL_DEV = False  # Leave disabled until the full development run is approved.
if not RUN_FULL_DEV:
    print('Full evaluation disabled. Stop after reviewing the pilot.')
else:
    assert sha256(DEV) == DEV_FILE_HASH and digest(dev_rows) == DEV_ROWS_HASH
    full_dir = PHASE2 / ('dev-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:8])
    full_args = list(pilot_args)
    if '--resume' in full_args:
        full_args.remove('--resume')
    full_args[full_args.index('--output') + 1] = str(full_dir)
    full_args[full_args.index('--mode') + 1] = 'full'
    print('Full development output:', full_dir)
    FULL_RESULTS = evaluate_dev(evaluation_parser().parse_args(full_args))
    print('Full development results:', FULL_RESULTS)
